In [4]:
%%writefile exp2.cu
#include <stdio.h>
#include <sys/time.h>
#define N 4
double cpuSecond()
{
struct timeval tp;
gettimeofday(&tp, NULL);
return ((double)tp.tv_sec + (double)tp.tv_usec * 1.0e-6);
}
__global__ void transpose(int *A, int *B) //global memory
{
__shared__ int tile[N][N]; //shared memory
int row = threadIdx.y;
int col = threadIdx.x;
tile[row][col] = A[row * N + col];
__syncthreads();
B[col * N + row] = tile[row][col];
}
int main()
{
int h_A[N][N] =
{

{1,2,3,4},
{5,6,7,8},
{9,10,11,12},
{13,14,15,16}
};
int h_B[N][N];
int *d_A, *d_B;
cudaMalloc((void**)&d_A, N * N * sizeof(int));
cudaMalloc((void**)&d_B, N * N * sizeof(int));
cudaMemcpy(d_A, h_A, N * N * sizeof(int), cudaMemcpyHostToDevice);
dim3 threads(N, N);
double start = cpuSecond();
transpose<<<1, threads>>>(d_A, d_B);
cudaDeviceSynchronize();
double end = cpuSecond();
cudaMemcpy(h_B, d_B, N * N * sizeof(int), cudaMemcpyDeviceToHost);
printf("Input Matrix\n");
for(int i = 0; i < N; i++)
{
for(int j = 0; j < N; j++)
printf("%4d", h_A[i][j]);
printf("\n");
}
printf("\nTranspose Matrix\n");
for(int i = 0; i < N; i++){
for(int j = 0; j < N; j++)
printf("%4d", h_B[i][j]);
printf("\n");
}
printf("\nExecution Time = %lf seconds\n", end - start);
cudaFree(d_A);
cudaFree(d_B);
return 0;
}

Writing exp2.cu


In [5]:
!nvcc exp2.cu -o exp2 && ./exp2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Input Matrix
   1   2   3   4
   5   6   7   8
   9  10  11  12
  13  14  15  16

Transpose Matrix
   1   5   9  13
   2   6  10  14
   3   7  11  15
   4   8  12  16

Execution Time = 0.086167 seconds


In [6]:
%%writefile exp4.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 1000000

__global__ void kernel(int *next, int *w)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N && next[i] != -1) {
        w[i] += w[next[i]];
        next[i] = next[next[i]];
    }
}

void cpu(int *next, int *w)
{
    for (int s = 0; s < 20; s++)
        for (int i = 0; i < N; i++)
            if (next[i] != -1) {
                w[i] += w[next[i]];
                next[i] = next[next[i]];
            }
}

int main()
{
    int *hn = (int*)malloc(N*sizeof(int));
    int *hw = (int*)malloc(N*sizeof(int));
    int *gn = (int*)malloc(N*sizeof(int));
    int *gw = (int*)malloc(N*sizeof(int));

    for (int i=0; i<N; i++) {
        hn[i] = gn[i] = (i+1<N) ? i+1 : -1;
        hw[i] = gw[i] = 1;
    }

    clock_t a = clock();
    cpu(hn,hw);
    double ct = (double)(clock()-a)/CLOCKS_PER_SEC*1000;

    int *dn,*dw;
    cudaMalloc(&dn,N*sizeof(int));
    cudaMalloc(&dw,N*sizeof(int));
    cudaMemcpy(dn,gn,N*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(dw,gw,N*sizeof(int),cudaMemcpyHostToDevice);

    cudaEvent_t a1,b1;
    cudaEventCreate(&a1);
    cudaEventCreate(&b1);

    cudaEventRecord(a1);

    for(int s=0;s<20;s++)
        kernel<<<(N+255)/256,256>>>(dn,dw);

    cudaEventRecord(b1);
    cudaEventSynchronize(b1);

    float gt;
    cudaEventElapsedTime(&gt,a1,b1);

    cudaMemcpy(gw,dw,N*sizeof(int),cudaMemcpyDeviceToHost);

    printf("CPU: %d | %.2f ms\n",hw[0],ct);
    printf("GPU: %d | %.2f ms\n",gw[0],gt);
    printf("Speedup: %.2fx\n",ct/gt);
}

Writing exp4.cu


In [7]:
!nvcc exp4.cu -o exp4 && ./exp4

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CPU: 1000000 | 115.37 ms
GPU: 752064 | 20.40 ms
Speedup: 5.65x


In [1]:
%%writefile exp5.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 30000

__global__ void sort(int *a, int n, int p)
{
    int k = blockIdx.x * blockDim.x + threadIdx.x;
    int i = 2*k + p%2;

    if(i+1 < n && a[i] > a[i+1]){
        int t=a[i];
        a[i]=a[i+1];
        a[i+1]=t;
    }
}

void cpu(int *a)
{
    for(int p=0;p<N;p++)
        for(int i=p%2;i+1<N;i+=2)
            if(a[i]>a[i+1]){
                int t=a[i];
                a[i]=a[i+1];
                a[i+1]=t;
            }
}

int main()
{
    int *h1=(int*)malloc(N*sizeof(int));
    int *h2=(int*)malloc(N*sizeof(int));

    for(int i=0;i<N;i++)
        h1[i]=h2[i]=N-i;

    clock_t s=clock();
    cpu(h1);
    double ct=(double)(clock()-s)/CLOCKS_PER_SEC*1000;

    int *d;
    cudaMalloc(&d,N*sizeof(int));
    cudaMemcpy(d,h2,N*sizeof(int),cudaMemcpyHostToDevice);

    cudaEvent_t a,b;
    cudaEventCreate(&a);
    cudaEventCreate(&b);

    cudaEventRecord(a);

    for(int p=0;p<N;p++)
        sort<<<(N/2+255)/256,256>>>(d,N,p);

    cudaEventRecord(b);
    cudaEventSynchronize(b);

    float gt;
    cudaEventElapsedTime(&gt,a,b);

    cudaMemcpy(h2,d,N*sizeof(int),cudaMemcpyDeviceToHost);

    printf("CPU: %d | %.2f ms\n",h1[0],ct);
    printf("GPU: %d | %.2f ms\n",h2[0],gt);
    printf("Speedup: %.2fx\n",ct/gt);
}

Writing exp5.cu


In [2]:
!nvcc exp5.cu -o exp5 && ./exp5

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CPU: 1 | 2596.13 ms
GPU: 1 | 189.40 ms
Speedup: 13.71x


In [3]:
%%writefile exp6.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 100000
#define B 256

__global__ void bitonic(int *a,int j,int k)
{
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    int x=i^j;

    if(i<N && x>i) {
        if((i&k)==0) {
            if(a[i]>a[x]) {
                int t=a[i]; a[i]=a[x]; a[x]=t;
            }
        }
        else {
            if(a[i]<a[x]) {
                int t=a[i]; a[i]=a[x]; a[x]=t;
            }
        }
    }
}

void quicksort(int *a,int l,int r)
{
    if(l>=r) return;

    int p=a[r],i=l-1;

    for(int j=l;j<r;j++)
        if(a[j]<=p) {
            i++;
            int t=a[i]; a[i]=a[j]; a[j]=t;
        }

    int t=a[i+1]; a[i+1]=a[r]; a[r]=t;
    int p1=i+1;

    quicksort(a,l,p1-1);
    quicksort(a,p1+1,r);
}

int main()
{
    int *h1=(int*)malloc(N*sizeof(int));
    int *h2=(int*)malloc(N*sizeof(int));

    for(int i=0;i<N;i++)
        h1[i]=h2[i]=rand()%10000;

    clock_t s=clock();
    quicksort(h1,0,N-1);
    double ct=(double)(clock()-s)/CLOCKS_PER_SEC*1000;

    int *d;
    cudaMalloc(&d,N*sizeof(int));
    cudaMemcpy(d,h2,N*sizeof(int),cudaMemcpyHostToDevice);

    cudaEvent_t a,b;
    cudaEventCreate(&a);
    cudaEventCreate(&b);

    cudaEventRecord(a);

    for(int k=2;k<=N;k<<=1)
        for(int j=k>>1;j>0;j>>=1)
            bitonic<<<(N+B-1)/B,B>>>(d,j,k);

    cudaEventRecord(b);
    cudaEventSynchronize(b);

    float gt;
    cudaEventElapsedTime(&gt,a,b);

    cudaMemcpy(h2,d,N*sizeof(int),cudaMemcpyDeviceToHost);

    printf("CPU: %d | %.2f ms\n",h1[0],ct);
    printf("GPU: %d | %.2f ms\n",h2[0],gt);
    printf("Speedup: %.2fx\n",ct/gt);
}

Writing exp6.cu


In [4]:
!nvcc -rdc=true exp6.cu -o exp6 && ./exp6

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CPU: 0 | 15.25 ms
GPU: 0 | 13.60 ms
Speedup: 1.12x


In [5]:
%%writefile exp3.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 10000000
#define B 256

__constant__ float f[3];

__global__ void stencil(float *in,float *out)
{
    int i=blockIdx.x*blockDim.x+threadIdx.x;

    if(i>0 && i<N-1)
        out[i]=in[i-1]*f[0]+in[i]*f[1]+in[i+1]*f[2];
}

void cpu(float *in,float *out,float *f)
{
    for(int i=1;i<N-1;i++)
        out[i]=in[i-1]*f[0]+in[i]*f[1]+in[i+1]*f[2];
}

int main()
{
    size_t s=N*sizeof(float);
    float f_h[3]={.25f,.5f,.25f};

    float *h1=(float*)malloc(s);
    float *h2=(float*)malloc(s);
    float *h3=(float*)malloc(s);

    for(int i=0;i<N;i++) h1[i]=1;

    clock_t a=clock();
    cpu(h1,h2,f_h);
    double ct=(double)(clock()-a)/CLOCKS_PER_SEC*1000;

    float *d1,*d2;
    cudaMalloc(&d1,s);
    cudaMalloc(&d2,s);
    cudaMemcpy(d1,h1,s,cudaMemcpyHostToDevice);
    cudaMemcpyToSymbol(f,f_h,3*sizeof(float));

    cudaEvent_t x,y;
    cudaEventCreate(&x);
    cudaEventCreate(&y);

    cudaEventRecord(x);
    stencil<<<(N+B-1)/B,B>>>(d1,d2);
    cudaEventRecord(y);
    cudaEventSynchronize(y);

    float gt;
    cudaEventElapsedTime(&gt,x,y);

    cudaMemcpy(h3,d2,s,cudaMemcpyDeviceToHost);

    printf("CPU: %.2f | %.2f ms\n",h2[10],ct);
    printf("GPU: %.2f | %.2f ms\n",h3[10],gt);
    printf("Speedup: %.2fx\n",ct/gt);
}

Writing exp3.cu


In [6]:
!nvcc exp3.cu -o exp3 && ./exp3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
CPU: 1.00 | 59.34 ms
GPU: 1.00 | 0.35 ms
Speedup: 168.13x
